In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ClusterDynamics – Unified Solver Notebook
# Ghoniem & Cho (1979) | 316 Stainless Steel | 450 °C | P = 1e-6 dpa/s
# ══════════════════════════════════════════════════════════════════════════════
#
# Select solver mode by setting MODE below:
# 
#   MODE               Backend           NV (default)  NI (default)  Wall time
#   ─────────────────  ────────────────  ────────────  ────────────  ─────────
#   'python_alone'     Python LSODA            50           100       seconds
#   'expanding_front'  CVODE Phase I         1,000        10,000      ~6 s
#   'dynamic_window'   CVODE Phase II        1,000       100,000      ~30 s
#   'sliding_window'   CVODE Phase III      10,000     1,000,000      ~78 s
#   'sliding_OpenMP'   CVODE Phase IV       10,000     1,000,000      ~80 s
#
# Phase descriptions:
#   python_alone    – scipy LSODA, segmented x_max-gated integration
#   expanding_front – CVODE BDF/GMRES, dynamic upper-truncation window (Phase I)
#   dynamic_window  – Phase I + lower QSS contraction + geometric expansion (Phase II)
#   sliding_window  – constant-width W window slides with cluster front (Phase III)
#   sliding_OpenMP  – Phase III + OpenMP + -O3 -march=native SIMD (Phase IV)
#

# ── 1. User selection ─────────────────────────────────────────────────────────
MODE = 'expanding_front'   # ← set desired solver mode here

# ── 2. Problem size (set to None to use the mode defaults in the table above) ─
NV = None    # maximum vacancy cluster size
NI = None    # maximum interstitial cluster size

# ── 3. Output and graph configuration ────────────────────────────────────────
SAVE_OUTPUT           = True   # save PNG figures to output/ directory
N_TIME_EVOL_CURVES    = 10     # time-evolution curves per panel (Figs 2–7)
N_SIZE_DIST_SNAPSHOTS = 10     # size-distribution snapshots per panel (Figs 8–13)

# ═════════════════════════════════════════════════════════════════════════════
# Nothing below this line needs to be changed for a standard run.
# ═════════════════════════════════════════════════════════════════════════════

import sys, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ── Path setup ────────────────────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    _NOTEBOOK_DIR = Path().resolve()

BASE_DIR   = _NOTEBOOK_DIR.parent      # ClusterDynamics/
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from py_utils.simulation    import ClusterDynamicsSimulation
from py_utils.cpp_bridge    import run_cpp_solver
from py_utils.visualization import plot_results

# ── Validate mode and apply defaults ─────────────────────────────────────────
_VALID_MODES = ('python_alone', 'expanding_front', 'dynamic_window',
                'sliding_window', 'sliding_OpenMP')
if MODE not in _VALID_MODES:
    raise ValueError(f'MODE={MODE!r} is not valid. Choose from: {_VALID_MODES}')

_DEFAULTS = {
    'python_alone':    {'NV':    50, 'NI':      100},
    'expanding_front': {'NV':  1000, 'NI':    10000},
    'dynamic_window':  {'NV':  1000, 'NI':   100000},
    'sliding_window':  {'NV': 10000, 'NI':  1000000},
    'sliding_OpenMP':  {'NV': 10000, 'NI':  1000000},
}
NV = NV or _DEFAULTS[MODE]['NV']
NI = NI or _DEFAULTS[MODE]['NI']

print('=' * 65)
print(f'ClusterDynamics  |  MODE = "{MODE}"')
print(f'Ghoniem & Cho (1979) | 316 SS | 450 °C | P=1e-6 dpa/s')
print(f'Problem size: Nv={NV:,}, Ni={NI:,}, N_EQ={NV+NI:,}')
print('=' * 65)

# ── Solver dispatch ───────────────────────────────────────────────────────────
sim = ClusterDynamicsSimulation(Nv=NV, Ni=NI)
t0  = time.perf_counter()

if MODE == 'python_alone':
    SOLVER_CONFIG = {
        't_span':     (1e-6, 1e6),
        'n_segments': 60,
        'rtol':       1e-4,
        'atol':       1e-20,
    }
    results = sim.run_simulation(
        t_span     = SOLVER_CONFIG['t_span'],
        n_segments = SOLVER_CONFIG['n_segments'],
        rtol       = SOLVER_CONFIG['rtol'],
        atol       = SOLVER_CONFIG['atol'],
        verbose    = True,
    )
    LABEL = 'Python LSODA (segmented, x_max-gated)'

elif MODE == 'expanding_front':
    # Phase I – dynamic upper-truncation window
    # Starts with a small active window and expands only when needed.
    # ~40× speedup vs full solver for Ni=10,000.
    SOLVER_CONFIG = {
        't_span':   (1e-3, 1e6),
        'n_points': 1000,
        'rtol':     1e-4,
        'atol':     1e-20,
        'log_time': True,
        'solver_method': {
            'backend':            'cvode',
            'lmm':                'bdf',
            'linsol':             'gmres',
            'window_mode':        1,
            'window_w0_v':        100,
            'window_w0_i':        100,
            'window_C_expand':    1e-20,
            'window_expand_pad':  20,
            'window_check_every': 1,
        },
    }
    results = run_cpp_solver(sim, SOLVER_CONFIG, base_dir=BASE_DIR)
    LABEL = 'C++ CVODE BDF Phase I (expanding front)'

elif MODE == 'dynamic_window':
    # Phase II – sliding window with lower QSS contraction
    # Upper expansion (geometric) + lower contraction (QSS criterion).
    # Jacobi diagonal preconditioner for GMRES. ~0.01% accuracy vs full solver.
    SOLVER_CONFIG = {
        't_span':   (1e-3, 1e6),
        'n_points': 1000,
        'rtol':     1e-4,
        'atol':     1e-20,
        'log_time': True,
        'solver_method': {
            'backend':              'cvode',
            'lmm':                  'bdf',
            'linsol':               'gmres',
            'window_mode':          2,
            'window_w0_v':          100,
            'window_w0_i':          100,
            'window_C_expand':      1e-20,
            'window_expand_pad':    20,
            'window_expand_factor': 2.0,
            'window_check_every':   20,
            'window_C_contract':    1e-3,
            'window_min_active_i':  100,
            'window_prec':          1,
        },
    }
    results = run_cpp_solver(sim, SOLVER_CONFIG, base_dir=BASE_DIR)
    LABEL = 'C++ CVODE BDF Phase II (dynamic window)'

elif MODE == 'sliding_window':
    # Phase III – constant-width sliding window
    # Fixed-width window W slides rigidly with the cluster front.
    # Nucleation guard at t < window_t_start prevents early lower-front sliding.
    # Scales to Ni = 1,000,000. ~6.9× speedup from -O3 -march=native (SIMD).
    SOLVER_CONFIG = {
        't_span':   (1e-5, 1e7),
        'n_points': 1000,
        'rtol':     1e-4,
        'atol':     1e-20,
        'log_time': True,
        'solver_method': {
            'backend':              'cvode',
            'lmm':                  'bdf',
            'linsol':               'gmres',
            'window_mode':          3,
            'window_w0_v':          500,
            'window_w0_i':          1000,
            'window_C_expand':      1e-18,
            'window_expand_pad':    1000,   # = window_width → advance by exactly W
            'window_expand_factor': 1.0,    # disable geometric doubling
            'window_check_every':   10,
            'window_width':         1000,
            'window_t_start':       10.0,
            'window_N_thresh':      2000,
            'window_prec':          1,
        },
    }
    results = run_cpp_solver(sim, SOLVER_CONFIG, base_dir=BASE_DIR)
    LABEL = 'C++ CVODE BDF Phase III (sliding window)'

else:  # 'sliding_OpenMP'
    # Phase IV – sliding window + OpenMP + SIMD
    # Three speedup layers: -O3 -march=native SIMD, pre-allocated scratch buffers,
    # and a single #pragma omp parallel region per RHS call.
    # NOTE: -ffast-math intentionally omitted — alters IEEE-754 and causes CVODE
    # error estimator to diverge on stiff problems.
    # OMP_MIN_WORK=20000 threshold auto-falls back to serial path when W < 20000.
    SOLVER_CONFIG = {
        't_span':   (1e-5, 1e7),
        'n_points': 1000,
        'rtol':     1e-4,
        'atol':     1e-20,
        'log_time': True,
        'solver_method': {
            'backend':              'cvode',
            'lmm':                  'bdf',
            'linsol':               'gmres',
            'window_mode':          4,
            'window_omp_threads':   10,    # 10 of 12 M3-Max P-cores
            'window_w0_v':          500,
            'window_w0_i':          1000,
            'window_C_expand':      1e-18,
            'window_expand_pad':    1000,   # = window_width → advance by exactly W
            'window_expand_factor': 1.0,    # disable geometric doubling
            'window_check_every':   10,
            'window_width':         1000,
            'window_t_start':       10.0,
            'window_N_thresh':      2000,
            'window_prec':          1,
        },
    }
    results = run_cpp_solver(sim, SOLVER_CONFIG, base_dir=BASE_DIR)
    LABEL = 'C++ CVODE BDF Phase IV (sliding window + OpenMP)'

elapsed = time.perf_counter() - t0
print(f'\nWall time: {elapsed:.1f} s')

# ── Plot configuration ────────────────────────────────────────────────────────
PLOT_CONFIG = {
    'point_defects': {
        'xlim': (1e-6, 1e7),
        'ylim': (1e-15, 1e-3),
    },
    'interstitial_clusters': {
        'xlim': {
            'small': (1e-6, 1e5),
            'mid':   (1e2,  1e7),
            'large': (1e2,  1e7),
        },
        'C_floor':  1e-25,
        'n_curves': N_TIME_EVOL_CURVES,
    },
    'vacancy_clusters': {
        'xlim': {
            'small': (1e-6, 1e5),
            'mid':   (1e2,  1e7),
            'large': (1e2,  1e7),
        },
        'C_floor':  1e-20,
        'n_curves': N_TIME_EVOL_CURVES,
    },
    'i_cluster_size': {
        'xlim':        None,
        'ylim':        None,
        'n_snapshots': N_SIZE_DIST_SNAPSHOTS,
    },
    'v_cluster_size': {
        'xlim':        None,
        'ylim':        None,
        'n_snapshots': N_SIZE_DIST_SNAPSHOTS,
    },
}

# ── Plots ─────────────────────────────────────────────────────────────────────
if results is not None:
    run_dir = plot_results(
        sim, results,
        output_dir  = str(OUTPUT_DIR),
        save_plots  = SAVE_OUTPUT,
        sim_config  = SOLVER_CONFIG,
        label       = LABEL,
        plot_config = PLOT_CONFIG,
        wall_time   = elapsed,
    )

# ── Summary diagnostics ───────────────────────────────────────────────────────
if results is not None:
    conc = results['concentrations']
    t    = results['time']
    print('\n' + '=' * 65)
    print('SUMMARY')
    print('=' * 65)
    print(f'  Mode:         {MODE}')
    print(f'  Nv / Ni:      {NV:,} / {NI:,}  (N_EQ = {NV+NI:,})')
    print(f'  t_final:      {t[-1]:.2e} s')
    print(f'  Wall time:    {elapsed:.1f} s')
    print(f'  Cv1(t_end):   {conc["Cv1"][-1]:.4e}')
    print(f'  Ci1(t_end):   {conc["Ci1"][-1]:.4e}')
    if MODE != 'python_alone' and 'active_band' in results:
        active = results['active_band']
        lo_i = int(active['x_min'][-1])
        hi_i = int(active['x_max'][-1])
        if hi_i > 0:
            print(f'  Active Ci:    [{lo_i}..{hi_i}]  ({hi_i/NI*100:.1f}% of Ni)')
        hi_v = next((x for x in range(NV, 0, -1)
                     if conc.get(f'Cv{x}', [0.0])[-1] > 0.0), 0)
        if hi_v > 0:
            print(f'  Active Cv:    [1..{hi_v}]')
    print(f'  Graphs:       {N_TIME_EVOL_CURVES} time-evol curves/panel  |  '
          f'{N_SIZE_DIST_SNAPSHOTS} size-dist snapshots/panel')
    if SAVE_OUTPUT and 'run_dir' in dir():
        print(f'  Output:       {run_dir}')
    print('=' * 65)
else:
    print(f'\n  Simulation FAILED for MODE="{MODE}"')


ClusterDynamics  |  MODE = "expanding_front"
Ghoniem & Cho (1979) | 316 SS | 450 °C | P=1e-6 dpa/s
Problem size: Nv=1,000, Ni=10,000, N_EQ=11,000
Initializing ClusterDynamics simulation…
InputData: T=723.15 K  Cv_eq=7.068e-12  alpha=9.691e+13
ReactionRates: KCV[0]=7.037e+01  KLI[0]=3.391e+12  GCV[0]=1.075e-02
RateEquations: N=11000 equations  (Nv=1000, Ni=10000)
✓ Setup validation complete.
✓ Simulation initialized successfully!
Running C++ solver (solver)  …  (Nv=1000, Ni=10000, N_EQ=11000, param_file)
C++ solver info:
[window] reinit #1  x_hi_v=100  x_hi_i=100  N_active=200  t0=0.001
Solver: CVODE BDF WINDOW | linsol: GMRES | w0_v=100 w0_i=100 | C_expand=1e-20 | expand_pad=20 | Nv=1000 Ni=10000
[window] reinit #2  x_hi_v=100  x_hi_i=120  N_active=220  t0=1000
[window] reinit #3  x_hi_v=100  x_hi_i=140  N_active=240  t0=1282.65
[window] reinit #4  x_hi_v=100  x_hi_i=160  N_active=260  t0=1645.19
[window] reinit #5  x_hi_v=100  x_hi_i=180  N_active=280  t0=1942.17
[window] reinit #6  x

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# PARAMETER OPTIMIZATION  –  sliding_window mode  |  NV=1000  NI=1e5
#
# Find combinations of (E_m_i, E_b_2i, Z_i, rho_d) that produce:
#   • Interstitial cluster number density : 1e19 – 1e20  m^-3
#   • Mean interstitial cluster diameter  : 5  – 30      nm
#
# Strategy: scipy differential_evolution on a log-penalised objective.
#   popsize=5  →  initial population of 4×5=20 evaluations
#   maxiter=8  →  up to ~60 total solver calls  (~30 min at ~30 s/call)
# ══════════════════════════════════════════════════════════════════════════════

import sys, io, time, warnings
import numpy as np
from scipy.optimize import differential_evolution
from pathlib import Path
warnings.filterwarnings('ignore')

# ── Path setup ────────────────────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    _NOTEBOOK_DIR = Path().resolve()

BASE_DIR = _NOTEBOOK_DIR.parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from py_utils.simulation    import ClusterDynamicsSimulation
from py_utils.reaction_rates import ReactionRates
from py_utils.rate_equations import RateEquations
from py_utils.cpp_bridge    import run_cpp_solver


# ══════════════════════════════════════════════════════════════════════════════
# 1. PROBLEM SIZE AND PHYSICAL TARGETS
# ══════════════════════════════════════════════════════════════════════════════

NV_OPT = 1000
NI_OPT = int(1e5)

TARGET_DENSITY_MIN = 1e19   # interstitial cluster density  [m^-3]
TARGET_DENSITY_MAX = 1e20
TARGET_DIAM_MIN    = 5.0    # mean cluster diameter [nm]
TARGET_DIAM_MAX    = 30.0

# ── Unit-conversion helpers ───────────────────────────────────────────────────
_a_nm   = 0.363                        # lattice parameter [nm]  (3.63 Å)
_n_atom = 4.0 / (_a_nm * 1e-9)**3     # FCC atomic number density [m^-3]

def _mean_diam_nm(mean_n):
    """
    Mean diameter [nm] of a circular Frank partial loop containing mean_n SIA.
    Burgers vector b = a/sqrt(3), loop area = n*Omega/b, Omega = a^3/4
      => r [nm] = a_nm * sqrt( n*sqrt(3) / (4*pi) )
    """
    return 2.0 * _a_nm * np.sqrt(mean_n * np.sqrt(3.0) / (4.0 * np.pi))

def _cluster_density_m3(total_i_frac, mean_n):
    """
    Convert atom-fraction totals to cluster number density [m^-3].
      total_i_frac = sum_x( x * Ci_x )   [dimensionless atom fraction]
      mean_n       = sum_x(x*Ci_x) / sum_x(Ci_x)
    => N_clusters/vol = (total_i_frac / mean_n) * n_atom
    """
    return (total_i_frac / max(mean_n, 1.0)) * _n_atom


# ══════════════════════════════════════════════════════════════════════════════
# 2. SLIDING-WINDOW SOLVER CONFIGURATION  (tuned for NV=1000, NI=1e5)
# ══════════════════════════════════════════════════════════════════════════════

OPT_SOLVER_CONFIG = {
    't_span':   (1e-5, 1e7),
    'n_points': 200,           # fewer output rows → faster I/O
    'rtol':     1e-4,
    'atol':     1e-20,
    'log_time': True,
    'solver_method': {
        'backend':              'cvode',
        'lmm':                  'bdf',
        'linsol':               'gmres',
        'window_mode':          3,      # Phase III – constant-width sliding window
        'window_w0_v':          200,
        'window_w0_i':          500,
        'window_C_expand':      1e-18,
        'window_expand_pad':    500,    # = window_width → advance by exactly W
        'window_expand_factor': 1.0,    # disable geometric doubling
        'window_check_every':   10,
        'window_width':         500,
        'window_t_start':       10.0,
        'window_N_thresh':      2000,
        'window_prec':          1,
    },
}


# ══════════════════════════════════════════════════════════════════════════════
# 3. PARAMETER SEARCH BOUNDS
#    rho_d is searched on a log10 scale to cover several decades uniformly.
# ══════════════════════════════════════════════════════════════════════════════

PARAM_BOUNDS = {
    'E_m_i':      (0.05, 0.50),          # [eV]
    'E_b_2i':     (0.10, 0.80),          # [eV]
    'Z_i':        (1.00, 1.30),          # [-]
    'log10_rho_d': (8.0, 11.0),          # log10 of rho_d [cm cm^-3]
}

_BOUNDS_LIST = [
    PARAM_BOUNDS['E_m_i'],
    PARAM_BOUNDS['E_b_2i'],
    PARAM_BOUNDS['Z_i'],
    PARAM_BOUNDS['log10_rho_d'],
]


# ══════════════════════════════════════════════════════════════════════════════
# 4. SINGLE-EVALUATION RUNNER
# ══════════════════════════════════════════════════════════════════════════════

def _run_one(E_m_i, E_b_2i, Z_i, rho_d):
    """
    Build a simulation with the given material parameters, run the C++ solver,
    and return (density [m^-3], mean_diam [nm]).
    Returns (None, None) on solver failure.
    """
    # Redirect noisy init printout to /dev/null during optimization
    _old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        sim = ClusterDynamicsSimulation(Nv=NV_OPT, Ni=NI_OPT)
    finally:
        sys.stdout = _old_stdout

    # Override material parameters and rebuild all derived quantities
    sim.input_data.material_params['E_m_i']  = float(E_m_i)
    sim.input_data.material_params['E_b_2i'] = float(E_b_2i)
    sim.input_data.material_params['Z_i']    = float(Z_i)
    sim.input_data.material_params['rho_d']  = float(rho_d)

    _old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        sim.input_data.calculate_derived_parameters()
        sim.reaction_rates = ReactionRates(sim.input_data)
        sim.rate_equations = RateEquations(sim.input_data, sim.reaction_rates)
        results = run_cpp_solver(sim, OPT_SOLVER_CONFIG, base_dir=BASE_DIR)
    finally:
        sys.stdout = _old_stdout

    if results is None:
        return None, None

    mean_n  = results['mean_sizes']['mean_i'][-1]    # interstitials / cluster
    total_i = results['totals']['total_i'][-1]        # atom fraction (all i-clusters)

    density = _cluster_density_m3(total_i, mean_n)
    diam    = _mean_diam_nm(mean_n)
    return density, diam


# ══════════════════════════════════════════════════════════════════════════════
# 5. OBJECTIVE FUNCTION  (penalty = 0 inside both target windows)
# ══════════════════════════════════════════════════════════════════════════════

_eval_log = []   # accumulates every evaluation for post-analysis

def _penalty(density, diam):
    """
    Log-scale penalty for density; linear-scale penalty for diameter.
    Both terms are zero while inside the target range.
    """
    if density is None or density <= 0:
        return 1e6

    log_d  = np.log10(density)
    log_lo = np.log10(TARGET_DENSITY_MIN)
    log_hi = np.log10(TARGET_DENSITY_MAX)
    p_dens = max(log_lo - log_d, 0.0)**2 + max(log_d - log_hi, 0.0)**2

    # Scale factor keeps both penalty components order-O(1) at 1-decade miss
    p_diam = (max(TARGET_DIAM_MIN - diam, 0.0)**2 +
              max(diam - TARGET_DIAM_MAX, 0.0)**2) * 0.01

    return p_dens + p_diam


def _objective(x):
    E_m_i, E_b_2i, Z_i, log_rho = x
    rho_d = 10.0**log_rho

    t0 = time.perf_counter()
    density, diam = _run_one(E_m_i, E_b_2i, Z_i, rho_d)
    elapsed = time.perf_counter() - t0

    pen = _penalty(density, diam)

    record = dict(
        E_m_i=E_m_i, E_b_2i=E_b_2i, Z_i=Z_i, rho_d=rho_d,
        density=density, diam_nm=diam,
        penalty=pen, wall_s=elapsed,
    )
    _eval_log.append(record)

    # Progress line
    n      = len(_eval_log)
    d_str  = f'{density:.2e}' if density else 'FAIL'
    dm_str = f'{diam:.1f}'    if diam    else 'N/A'
    flag   = 'IN RANGE' if pen == 0.0 else f'pen={pen:.3f}'
    print(f"  [{n:3d}] E_mi={E_m_i:.3f} E_b2i={E_b_2i:.3f} "
          f"Zi={Z_i:.3f} log_rho={log_rho:.2f}  ->  "
          f"rho_i={d_str} m^-3  d={dm_str} nm  {flag}  ({elapsed:.1f}s)")

    return pen


# ══════════════════════════════════════════════════════════════════════════════
# 6. RUN DIFFERENTIAL EVOLUTION
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 75)
print('PARAMETER OPTIMIZATION  |  sliding_window  |  NV=1000  NI=1e5')
print(f'Targets : rho_i in [1e19, 1e20] m^-3   mean_diam in [5, 30] nm')
print(f'Search  : E_m_i    in {PARAM_BOUNDS["E_m_i"]} eV')
print(f'          E_b_2i   in {PARAM_BOUNDS["E_b_2i"]} eV')
print(f'          Z_i      in {PARAM_BOUNDS["Z_i"]}')
print(f'          log10(rho_d) in {PARAM_BOUNDS["log10_rho_d"]}  [cm^-2]')
print('Optimizer: differential_evolution  popsize=5  maxiter=8')
print('=' * 75)

_eval_log.clear()
t_opt_start = time.perf_counter()

opt_result = differential_evolution(
    _objective,
    bounds   = _BOUNDS_LIST,
    strategy = 'best1bin',
    maxiter  = 8,
    popsize  = 5,      # 4 params * 5 = 20 initial population
    tol      = 0.01,   # stop early if std(population penalties) < tol * mean
    seed     = 42,
    workers  = 1,      # must be serial – CVODE subprocess is not fork-safe
    disp     = False,
    polish   = False,
)

t_opt_total = time.perf_counter() - t_opt_start


# ══════════════════════════════════════════════════════════════════════════════
# 7. RESULTS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 75)
print('OPTIMIZATION COMPLETE')
print(f'  Total evaluations : {len(_eval_log)}')
print(f'  Total wall time   : {t_opt_total/60:.1f} min')
print(f'  Final penalty     : {opt_result.fun:.4f}')
print('=' * 75)

best_evals = sorted(_eval_log, key=lambda r: r['penalty'])

print('\nTop 5 parameter sets (sorted by penalty):')
hdr = (f"{'#':>3}  {'E_m_i':>7}  {'E_b_2i':>7}  {'Z_i':>6}  "
       f"{'log_rho':>8}  {'rho_i [m-3]':>12}  {'d [nm]':>8}  {'penalty':>8}")
print(hdr)
print('-' * len(hdr))
for rank, r in enumerate(best_evals[:5], 1):
    d_str  = f'{r["density"]:.2e}' if r['density'] else 'FAIL'
    dm_str = f'{r["diam_nm"]:.1f}' if r['diam_nm'] else 'N/A'
    print(f"{rank:3d}  {r['E_m_i']:7.4f}  {r['E_b_2i']:7.4f}  {r['Z_i']:6.4f}  "
          f"{np.log10(r['rho_d']):8.3f}  {d_str:>12}  {dm_str:>8}  {r['penalty']:8.4f}")

best = best_evals[0]
print()
if best['penalty'] == 0.0:
    print('TARGET ACHIEVED. Best parameters:')
else:
    print('Best parameters found (targets partially met):')

print(f"  E_m_i  = {best['E_m_i']:.4f} eV          (baseline: 0.9 eV)")
print(f"  E_b_2i = {best['E_b_2i']:.4f} eV          (baseline: 0.4 eV)")
print(f"  Z_i    = {best['Z_i']:.4f}                (baseline: 1.08)")
print(f"  rho_d  = {best['rho_d']:.3e} cm^-2    (baseline: 1e9)")
print()
d_in  = (TARGET_DENSITY_MIN <= best['density'] <= TARGET_DENSITY_MAX) if best['density'] else False
dm_in = (TARGET_DIAM_MIN    <= best['diam_nm'] <= TARGET_DIAM_MAX)    if best['diam_nm']  else False
print(f"  => rho_i = {best['density']:.2e} m^-3   {'[IN RANGE]' if d_in else '[OUT OF RANGE]'}"
      f"   target: [1e19, 1e20]")
print(f"  => d_mean= {best['diam_nm']:.1f} nm          {'[IN RANGE]' if dm_in else '[OUT OF RANGE]'}"
      f"   target: [5, 30] nm")


PARAMETER OPTIMIZATION  |  sliding_window  |  NV=1000  NI=1e5
Targets : rho_i in [1e19, 1e20] m^-3   mean_diam in [5, 30] nm
Search  : E_m_i    in (0.05, 0.5) eV
          E_b_2i   in (0.1, 0.8) eV
          Z_i      in (1.0, 1.3)
          log10(rho_d) in (8.0, 11.0)  [cm^-2]
Optimizer: differential_evolution  popsize=5  maxiter=8
  [  1] E_mi=0.396 E_b2i=0.508 Zi=1.267 log_rho=8.64  ->  rho_i=5.75e+19 m^-3  d=19.1 nm  IN RANGE  (863.3s)
  [  2] E_mi=0.136 E_b2i=0.679 Zi=1.170 log_rho=9.42  ->  rho_i=1.17e+17 m^-3  d=15.6 nm  pen=3.725  (378.6s)
  [  3] E_mi=0.278 E_b2i=0.140 Zi=1.114 log_rho=8.28  ->  rho_i=3.41e+37 m^-3  d=0.5 nm  pen=307.606  (5.4s)
  [  4] E_mi=0.455 E_b2i=0.280 Zi=1.204 log_rho=10.30  ->  rho_i=3.64e+36 m^-3  d=0.5 nm  pen=274.479  (5.4s)
  [  5] E_mi=0.367 E_b2i=0.212 Zi=1.011 log_rho=9.08  ->  rho_i=1.07e+37 m^-3  d=0.5 nm  pen=290.269  (5.4s)
  [  6] E_mi=0.332 E_b2i=0.730 Zi=1.145 log_rho=8.80  ->  rho_i=5.97e+18 m^-3  d=18.9 nm  pen=0.050  (971.5s)
  [  7] E